# ProGen2 Protein Sequence Generation and Scoring

![ProGen2 Protein Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/progen2/hero.png)

This notebook demonstrates both functions of the ProGen2 tool. In the first section, we use `run_progen2_sample` to generate protein sequences from amino acid prompts using autoregressive sampling. In the second section, we use `run_progen2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model, enabling ranking and filtering of design candidates.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen2")
display_overview("progen2")
display_docs_section("progen2", "Background")

## Available tools

In [ ]:
display_available_tools("progen2")

### `run_progen2_sample`

ProGen2 generates protein sequences autoregressively from a prompt, extending the sequence one residue at a time according to its learned distribution over natural proteins. Available checkpoints range from 151M to 6B parameters, with a specialized OAS variant for antibody sequences. The `temperature` and `top_p` parameters control sampling diversity: lower temperature produces more conservative, high-confidence extensions while higher values explore more of sequence space.

In [ ]:
from proto_tools import (
    ProGen2SampleInput,
    ProGen2SampleConfig,
    run_progen2_sample,
)

In [ ]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_sample")

# Create a ProGen2 input with an N-terminal amino acid prompt
inputs = ProGen2SampleInput(prompts=["MKTAYIAKQRQISF"])

In [ ]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_sample")

# Configure sampling parameters | see docs above for all fields
config = ProGen2SampleConfig(
    model_checkpoint="progen2-large",
    max_new_tokens=120,
    temperature=0.8,
    top_p=0.95,
    top_k=0,
    truncate_at_stop=True,
    strip_special_tokens=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
sample_result = run_progen2_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} residues")

### `run_progen2_score`

ProGen2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each residue is given its preceding context. The resulting perplexity summarizes sequence naturalness: lower perplexity indicates the sequence more closely matches the statistical patterns of natural proteins in the training data. This is useful for ranking designed candidates, filtering out sequences unlikely to fold or function, or comparing wild-type and mutant variants.

In [ ]:
from proto_tools import (
    ProGen2ScoringInput,
    ProGen2ScoringConfig,
    run_progen2_score,
)

In [ ]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_score")

# Score two protein sequences
score_inputs = ProGen2ScoringInput(sequences=["MVLSPADKTN", "MKTLLILAVVAA"])

In [ ]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_score")

# Configure scoring | see docs above for all fields
score_config = ProGen2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_progen2_score(score_inputs, score_config)

In [ ]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="progen2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen2_sequences.fasta'}")

score_result.export(name="progen2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen2_scores.csv'}")